In [28]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score


In [16]:
df = pd.read_csv("cropdata_updated.csv")
print(f"Raw Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
df.head(10)

Raw Dataset Shape: 16411 rows, 7 columns



,crop ID,soil_type,Seedling Stage,MOI,temp,humidity,result
0,Wheat,Black Soil,Germination,1,25,80.0,1
1,Wheat,Black Soil,Germination,2,26,77.0,1
2,Wheat,Black Soil,Germination,3,27,74.0,1
3,Wheat,Black Soil,Germination,4,28,71.0,1
4,Wheat,Black Soil,Germination,5,29,68.0,1
5,Wheat,Black Soil,Germination,6,30,65.0,1
6,Wheat,Black Soil,Germination,7,31,62.0,1
7,Wheat,Black Soil,Germination,8,32,59.0,1
8,Wheat,Black Soil,Germination,9,33,56.0,1
9,Wheat,Black Soil,Germination,10,34,53.0,1


In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16411 entries, 0 to 16410
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   crop ID         16411 non-null  object 
 1   soil_type       16411 non-null  object 
 2   Seedling Stage  16411 non-null  object 
 3   MOI             16411 non-null  int64  
 4   temp            16411 non-null  int64  
 5   humidity        16411 non-null  float64
 6   result          16411 non-null  int64  
dtypes: float64(1), int64(3), object(3)
memory usage: 897.6+ KB


In [18]:
df.drop_duplicates(inplace=True)
df.shape

(16283, 7)

In [19]:
df = df[df["result"] != 2]
df.shape

(15161, 7)

In [20]:
target_counts = df['result'].value_counts().sort_index()
target_pct = df['result'].value_counts(normalize=True).sort_index() * 100

target_summary = pd.DataFrame({
    'Count': target_counts,
    'Percentage (%)': target_pct.round(2),
    'Description': [
        '0: No Irrigation',
        '1: Standard Irrigation',
    ]
})
target_summary

,Count,Percentage (%),Description
result,,,
0,8934,58.93,0: No Irrigation
1,6227,41.07,1: Standard Irrigation


In [21]:
df.rename(columns={
    "crop ID": "crop_name",
    "soil_type": "soil_type",
    "Seedling Stage": "seedling_stage",
    "MOI": "moi",
    "temp": "temp",
    "humidity": "humidity",
    "result": "result",
}, inplace=True)
df.head()

,crop_name,soil_type,seedling_stage,moi,temp,humidity,result
0,Wheat,Black Soil,Germination,1,25,80.0,1
1,Wheat,Black Soil,Germination,2,26,77.0,1
2,Wheat,Black Soil,Germination,3,27,74.0,1
3,Wheat,Black Soil,Germination,4,28,71.0,1
4,Wheat,Black Soil,Germination,5,29,68.0,1


In [22]:
numerical_cols = ['moi', 'temp', 'humidity']

for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]

    print(f"The column {col} has {len(outliers)} outliers that has a percentage of {round((len(outliers) / len(df)) * 100, 2)}%")



The column moi has 0 outliers that has a percentage of 0.0%
The column temp has 0 outliers that has a percentage of 0.0%
The column humidity has 0 outliers that has a percentage of 0.0%


In [23]:
GROWTH_STAGE_ORDER_MAP = {
    "Germination": 1,
    "Seedling Stage": 2,
    "Vegetative Growth / Root or Tuber Development": 3,
    "Flowering": 4,
    "Pollination": 5,
    "Fruit/Grain/Bulb Formation": 6,
    "Maturation": 7,
    "Harvest": 8,
}
df["growth_stage_order"] = (df["seedling_stage"].map(GROWTH_STAGE_ORDER_MAP).fillna(0).astype(int))

In [24]:
df["moi_temp_ratio"] = (df["moi"] / df["temp"].replace(0, np.nan)).round(4)
df["moi_humidity_index"] = ((df["moi"] * df["humidity"]) / 100.0).round(4)
df.shape

(15161, 10)

In [ ]:
df.to_csv("./data/cleaned_data.csv", index=False)

In [25]:
numerical_cols = ["moi", "temp", "humidity", "growth_stage_order", "moi_temp_ratio", "moi_humidity_index"]
categorial_cols = ["crop_name", "soil_type", "seedling_stage"]
feature_cols = categorial_cols + numerical_cols
X = df[feature_cols]
y = df['result']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [26]:
processor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(),["moi", "temp", "humidity", "moi_temp_ratio", "moi_humidity_index"]),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorial_cols)
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

In [27]:
X_train_transformed = processor.fit_transform(X_train)
X_test_transformed = processor.transform(X_test)
feature_names = processor.get_feature_names_out()

X_train_transformed = pd.DataFrame(X_train_transformed, columns=feature_names)
X_test_transformed = pd.DataFrame(X_test_transformed, columns=feature_names)

In [29]:
log_reg = LogisticRegression(max_iter=1000, random_state=42)


param_grid_log_reg = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']
}

grid_log_reg = GridSearchCV(
    estimator=log_reg,
    param_grid=param_grid_log_reg,
    cv=5,                 # 5-Fold Cross Validation
    scoring='accuracy',
    n_jobs=-1
)

grid_log_reg.fit(X_train_transformed, y_train)

print(f"best hyperparameters Logistic Regression: {grid_log_reg.best_params_}")
print(f"best accuracy CV: {grid_log_reg.best_score_:.4f}")


best_log_reg = grid_log_reg.best_estimator_
y_pred_log = best_log_reg.predict(X_test_transformed)
print("Accuracy Logistic Regression:", accuracy_score(y_test, y_pred_log))

best hyperparameters Logistic Regression: {'C': 1, 'penalty': 'l1', 'solver': 'liblinear'}
best accuracy CV: 0.9454
Accuracy Logistic Regression: 0.9436201780415431


In [31]:
rf_clf = RandomForestClassifier(random_state=42)


param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'criterion': ['gini', 'entropy']
}

grid_rf = GridSearchCV(
    estimator=rf_clf,
    param_grid=param_grid_rf,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)


grid_rf.fit(X_train_transformed, y_train)

print(f"best hyperparameters Random Forest: {grid_rf.best_params_}")
print(f"best accuracy  CV: {grid_rf.best_score_:.4f}")


best_rf = grid_rf.best_estimator_
y_pred_rf = best_rf.predict(X_test_transformed)
print("Accuracy Random Forest:", accuracy_score(y_test, y_pred_rf))

best hyperparameters Random Forest: {'criterion': 'entropy', 'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 50}
best accuracy  CV: 0.9985
Accuracy Random Forest: 0.9993405868776789


In [32]:
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1787
           1       1.00      1.00      1.00      1246

    accuracy                           1.00      3033
   macro avg       1.00      1.00      1.00      3033
weighted avg       1.00      1.00      1.00      3033

